# New York State Quarterly Private-Sector Compensation Forecasting by Industry

This notebook presents a fully reproducible applied time-series forecasting study using BEA-style quarterly compensation data in `data.csv`. It evaluates statistical and machine-learning model families for four core New York industries with policy and budgeting relevance.

## 1. Project Motivation and Policy Context
Compensation forecasting helps state agencies, budget offices, and economic planners anticipate wage-bill pressures, labor-market momentum, and sectoral income dynamics. In New York, Finance and insurance, Health care, Professional and scientific services, and Technology are economically influential yet structurally different industries.

These sectors differ in cyclicality, regulation exposure, labor intensity, and response to macro shocks. A transparent multi-model framework helps identify robust forecasting approaches for differing data-generating processes, supporting planning scenarios and fiscal surveillance.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


## 2. Data Cleaning and Reshaping

In [ ]:
TARGET_INDUSTRIES = {
    'Finance and insurance': 'Finance and insurance',
    'Health care and social assistance': 'Health care',
    'Professional, scientific, and technical services': 'Professional and scientific services',
    'Information': 'Technology'
}

TRAIN_END = pd.Period('2023Q3', freq='Q')
TEST_START = pd.Period('2023Q4', freq='Q')
TEST_END = pd.Period('2025Q3', freq='Q')
START_PERIOD = pd.Period('1998Q1', freq='Q')
END_PERIOD = pd.Period('2025Q3', freq='Q')

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def clean_bea_data(path='data.csv'):
    raw = pd.read_csv(path, skiprows=3)
    raw.columns = [c.strip() for c in raw.columns]
    if 'GeoName' in raw.columns:
        raw = raw[raw['GeoName'].astype(str).str.strip() == 'New York'].copy()
    raw['Description'] = raw['Description'].astype(str).str.strip()

    quarter_cols = [c for c in raw.columns if ':' in c and 'Q' in c]
    long_df = raw.melt(
        id_vars=['Description'],
        value_vars=quarter_cols,
        var_name='quarter',
        value_name='compensation'
    )
    long_df['compensation'] = pd.to_numeric(long_df['compensation'], errors='coerce')
    long_df['period'] = pd.PeriodIndex(long_df['quarter'].str.replace(':', ''), freq='Q')

    long_df = long_df[long_df['Description'].isin(TARGET_INDUSTRIES.keys())].copy()
    long_df['industry'] = long_df['Description'].map(TARGET_INDUSTRIES)
    long_df = long_df[['industry', 'period', 'compensation']]
    long_df = long_df[(long_df['period'] >= START_PERIOD) & (long_df['period'] <= END_PERIOD)]
    long_df = long_df.sort_values(['industry', 'period']).reset_index(drop=True)

    wide = long_df.pivot(index='period', columns='industry', values='compensation').sort_index()
    return raw, long_df, wide

raw_df, long_df, wide_df = clean_bea_data('data.csv')
display(raw_df.head())
display(long_df.head())
display(wide_df.head())

In [ ]:
# Validation checklist
print('Target industries in cleaned data:', sorted(long_df['industry'].unique()))
print('Coverage:', long_df['period'].min(), 'to', long_df['period'].max())
print('Train end:', TRAIN_END, '| Test start:', TEST_START, '| Test end:', TEST_END)

expected_periods = pd.period_range(START_PERIOD, END_PERIOD, freq='Q')
for ind in sorted(long_df['industry'].unique()):
    p = long_df.loc[long_df['industry'] == ind, 'period']
    assert len(p) == len(expected_periods), f'Missing periods for {ind}'
    assert set(p) == set(expected_periods), f'Quarter continuity failed for {ind}'

display(long_df.groupby('industry')['compensation'].agg(['count','min','max','mean','std']))
print('Missing values in panel:', wide_df.isna().sum().sum())

## 3. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for col in wide_df.columns:
    ax.plot(wide_df.index.to_timestamp(), wide_df[col], label=col)
ax.set_title('Quarterly Compensation Levels by Industry (New York)')
ax.set_xlabel('Quarter')
ax.set_ylabel('Compensation')
ax.legend()
plt.tight_layout()

In [ ]:
indexed = wide_df / wide_df.iloc[0] * 100
rolling_mean = wide_df.rolling(4).mean()
rolling_std = wide_df.pct_change().rolling(4).std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for c in indexed.columns: axes[0].plot(indexed.index.to_timestamp(), indexed[c], label=c)
axes[0].set_title('Indexed Growth (1998Q1=100)')
axes[0].legend()
for c in rolling_std.columns: axes[1].plot(rolling_std.index.to_timestamp(), rolling_std[c], label=c)
axes[1].set_title('4-Quarter Rolling Volatility of Growth')
axes[1].legend()
plt.tight_layout()

## 4. Stationarity and Diagnostics

In [ ]:
adf_rows = []
for col in wide_df.columns:
    s = wide_df[col].dropna()
    d = s.diff().dropna()
    adf_rows.append((col, 'level', adfuller(s)[0], adfuller(s)[1]))
    adf_rows.append((col, 'first_diff', adfuller(d)[0], adfuller(d)[1]))
adf_df = pd.DataFrame(adf_rows, columns=['industry','series','adf_stat','p_value'])
display(adf_df)

In [ ]:
sample = wide_df.columns[0]
decomp = seasonal_decompose(wide_df[sample], model='additive', period=4)
decomp.plot(); plt.suptitle(f'Seasonal Decomposition: {sample}', y=1.02); plt.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(wide_df[sample].dropna(), lags=20, ax=axes[0])
plot_pacf(wide_df[sample].dropna(), lags=20, ax=axes[1], method='ywm')
plt.tight_layout()

## 5–7. Forecasting Models (Exponential Smoothing, ARIMA-family, Random Forest)

In [ ]:
def split_series(s):
    train = s[s.index <= TRAIN_END]
    test = s[(s.index >= TEST_START) & (s.index <= TEST_END)]
    return train, test

def fit_exp_models(train, horizon):
    out = {}
    out['SES'] = SimpleExpSmoothing(train).fit().forecast(horizon)
    out['Holt'] = Holt(train).fit().forecast(horizon)
    for trend, seasonal, name in [('add','add','HW_add_add'), ('add','mul','HW_add_mul')]:
        try:
            m = ExponentialSmoothing(train, trend=trend, seasonal=seasonal, seasonal_periods=4).fit()
            out[name] = m.forecast(horizon)
        except Exception:
            pass
    return out

def fit_arima_candidates(train, horizon):
    candidates = {
        'AR(2)': (2,0,0), 'MA(2)': (0,0,2), 'ARMA(2,2)': (2,0,2),
        'ARIMA(1,1,1)': (1,1,1), 'ARIMA(2,1,2)': (2,1,2)
    }
    fcsts, diag = {}, []
    for name, order in candidates.items():
        try:
            m = ARIMA(train, order=order).fit()
            fcsts[name] = m.forecast(horizon)
            lb_p = acorr_ljungbox(m.resid.dropna(), lags=[8], return_df=True)['lb_pvalue'].iloc[0]
            diag.append([name, order, m.aic, m.bic, lb_p])
        except Exception:
            continue
    diag_df = pd.DataFrame(diag, columns=['model','order','aic','bic','ljung_box_p_lag8'])
    return fcsts, diag_df

def build_rf_features(s, max_lag=4):
    df = pd.DataFrame({'y': s.copy()})
    for l in range(1, max_lag + 1): df[f'lag_{l}'] = df['y'].shift(l)
    df['roll_mean_4'] = df['y'].shift(1).rolling(4).mean()
    df['roll_std_4'] = df['y'].shift(1).rolling(4).std()
    df['q'] = df.index.quarter
    qd = pd.get_dummies(df['q'], prefix='q', drop_first=True)
    df = pd.concat([df.drop(columns=['q']), qd], axis=1)
    df['trend'] = np.arange(len(df))
    return df

def recursive_rf_forecast(train, test_index):
    history = train.copy()
    preds = []
    for step, p in enumerate(test_index, 1):
        full = build_rf_features(history)
        full = full.dropna()
        X = full.drop(columns=['y'])
        y = full['y']
        rf = RandomForestRegressor(n_estimators=400, random_state=42)
        rf.fit(X, y)

        temp = history.copy()
        temp.loc[p] = np.nan
        feat_row = build_rf_features(temp).loc[p].drop(labels=['y'])
        pred = rf.predict(feat_row.values.reshape(1, -1))[0]
        preds.append(pred)
        history.loc[p] = pred
    return pd.Series(preds, index=test_index)


In [ ]:
all_forecasts = {}
all_metrics = []
all_arima_diag = {}

for ind in wide_df.columns:
    s = wide_df[ind].dropna()
    train, test = split_series(s)
    h = len(test)

    fcst = {}
    fcst.update(fit_exp_models(train, h))
    arima_fcst, arima_diag = fit_arima_candidates(train, h)
    fcst.update(arima_fcst)
    fcst['RandomForest'] = recursive_rf_forecast(train, test.index)

    all_forecasts[ind] = {'actual': test, 'models': fcst}
    all_arima_diag[ind] = arima_diag

    for model_name, pred in fcst.items():
        pred = pd.Series(pred, index=test.index)
        all_metrics.append({
            'industry': ind, 'model': model_name,
            'MAE': mean_absolute_error(test, pred),
            'MAPE': mape(test, pred),
            'R2': r2_score(test, pred)
        })

metrics_df = pd.DataFrame(all_metrics).sort_values(['industry', 'MAE'])
display(metrics_df)
display(metrics_df.loc[metrics_df.groupby('industry')['MAE'].idxmin()].sort_values('MAE'))

## 8–9. Forecast Evaluation and Residual Diagnostics

In [ ]:
for ind, pack in all_forecasts.items():
    actual = pack['actual']
    best_model = metrics_df[metrics_df['industry']==ind].sort_values('MAE').iloc[0]['model']
    pred = pd.Series(pack['models'][best_model], index=actual.index)
    resid = actual - pred

    fig, axes = plt.subplots(1, 3, figsize=(16,4))
    axes[0].plot(actual.index.to_timestamp(), actual, label='Actual')
    axes[0].plot(pred.index.to_timestamp(), pred, label=f'Forecast ({best_model})')
    axes[0].set_title(f'{ind}: Actual vs Forecast')
    axes[0].legend()

    axes[1].plot(resid.index.to_timestamp(), resid)
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].set_title('Residual Time Series')

    axes[2].hist(resid, bins=8)
    axes[2].set_title('Residual Histogram')
    plt.tight_layout()

    lb = acorr_ljungbox(resid, lags=[4], return_df=True)
    print(f'[{ind}] Best model: {best_model} | Ljung-Box p-value (lag4):', float(lb['lb_pvalue'].iloc[0]))

## 10. Final Conclusions
The holdout-only comparison (`2023Q4`–`2025Q3`) identifies best-in-class models by industry under consistent no-leakage rules. Results typically show that model fit varies across sectors due to differing trend strength, seasonality, and shock persistence.

Key interpretation guidance:
- Prefer simpler models when diagnostics indicate residual white-noise and stable holdout error.
- Prefer richer seasonal/trend structures in sectors with persistent quarterly patterns.
- Use model disagreement as an uncertainty signal for planning and scenario analysis.

## 11. Streamlit Dashboard Planning (No App Built)
This notebook can directly support a future Streamlit dashboard by exporting: (1) cleaned panel data, (2) holdout forecasts by model and industry, (3) metrics and rankings, and (4) diagnostics tables (AIC/BIC/Ljung-Box).

Recommended dashboard components:
- Industry/model selectors for interactive forecast and residual views.
- Parameterized forecast horizon controls (e.g., 4, 8, 12 quarters).
- KPI tiles for MAE, MAPE, and R² computed on a specified evaluation window.
- Download options for forecast tables and diagnostics for budgeting workflows.

Stakeholder value: policy teams can compare sector trajectories, budgeting teams can stress-test labor cost scenarios, and economic analysts can monitor model stability over time.